# X3: Fixed-dimensional MCMC with Eryn

**Development-stack exercise notebook (LATW `dev` branch).** There is no
Colab button: these exercises target the *development* versions of the LISA
Analysis Tools packages. Set the environment up by cloning LISAanalysistools
and running its installer (it lays every sibling repo out side by side and
editable-installs the development branches):

```bash
git clone https://github.com/lisa-analysis-tools/lisa-analysis-tools.git LISAanalysistools
bash LISAanalysistools/install.sh
```

For the workshop on the **pip-released** packages, use the
[`main` branch](https://github.com/lisa-analysis-tools/LATW/tree/main) instead
(branch policy: `main` &harr; pip releases, `dev` &harr; the `install.sh` stack).

In [ ]:
import os

# Threading is pinned to 1 everywhere in this workshop (MPI-only policy;
# OMP-threaded kernels have caused out-of-memory kills on laptops).
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")

import warnings
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from copy import deepcopy
from lisatools.utils.constants import *

# Eryn's internals still import a legacy prior module (harmless) and the noise
# models divide by f=0 on full grids; silence both so the output stays clean.
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [`X1`](X1_SensitivitySNR.ipynb) we learned to score a single point in
parameter space &mdash; an inner product, an SNR, a likelihood. This notebook
turns that single-point likelihood into a **posterior** by sampling. We build a
Metropolis sampler by hand to see the mechanism, reproduce it with
[Eryn](https://lisa-analysis-tools.github.io/Eryn)'s `EnsembleSampler`, add **parallel
tempering** so the sampler can cross between isolated modes, and finish by
recovering the parameters of a small GW-flavored signal &mdash; using the very
`AnalysisContainer` from `X1` as the sampler's log-likelihood. The companion
informational notebook is [`06`](../../06_ErynSmallToLarge.ipynb); everything
here is fixed-dimension (the model does not add or remove parameters) and runs
on the laptop CPU in a few minutes.

### How these exercises work

Each exercise is one of two kinds:

- **Task N** &mdash; you *write code* toward a stated goal. In this answer
  notebook the solution cells are filled in; in the generated student notebook
  they are blanked (a whole cell, or just the key solution lines for a
  fill-in-the-blank). Every Task ends with a **Useful documentation:** list
  pointing at the Sphinx/Eryn API docs and the relevant informational-notebook
  section.
- **Question** (a `### Question` heading) &mdash; a short *discussion* prompt.
  No code required; the answer sketch here is for the group conversation and is
  removed in the student notebook.

The tasks build on each other in order, so run them top to bottom.

## Task 1: Build your own Metropolis sampler

Before reaching for a library, implement the core of MCMC yourself. The
target is a one-dimensional unit Gaussian ($\mu=0$, $\sigma=1$) with a flat
(uniform) prior, so the posterior is just that Gaussian. The
**Metropolis&ndash;Hastings** recipe is a short loop: from the current point,
*propose* a nearby point, *evaluate* the log-likelihood there, and *accept* the
move with probability $\min(1, e^{\Delta\ln\mathcal{P}})$ &mdash; which, with
a uniform prior, means accept whenever
$\Delta\ln\mathcal{L} > \ln U(0,1)$.

The log-likelihood and the loop scaffolding are given; **fill in the three
marked lines** (propose / evaluate / accept). Then plot the chain (you will see
an initial *burn-in* transient) and the histogram of the post-burn-in samples
against the target density.

Useful documentation:
* [`numpy.random`](https://numpy.org/doc/stable/reference/random/index.html)
  (`randn`, `rand`, `uniform`)
* Informational notebook: see [`06` &sect; A fixed-dimension ensemble sampler](../../06_ErynSmallToLarge.ipynb)
  (Eryn automates exactly this loop, run over a population of walkers)

In [ ]:
# provided: the target log-likelihood + a look at the density
def log_like_gauss(x):
    # standard-normal log-likelihood (mu = 0, sigma = 1), up to a constant
    return -0.5 * x ** 2 - 0.5 * np.log(2 * np.pi)

xg = np.linspace(-6.0, 6.0, 500)
plt.plot(xg, np.exp(log_like_gauss(xg)))
plt.axvline(0.0, color="k", lw=0.5)
plt.xlabel("x"); plt.ylabel(r"$\mathcal{L}(x)$"); plt.title("target: unit Gaussian")
plt.show()

Fill in the three lines marked below. (In the student notebook they are
blank; the surrounding loop, comments, and bookkeeping are kept.)

In [ ]:
num_steps = 50000

# random starting point drawn from the (uniform) prior range
current = np.random.uniform(-10.0, 10.0)
current_ll = log_like_gauss(current)
chain = np.empty(num_steps)

for step in range(num_steps):
    # 1) propose a new point: current + a Gaussian step of width 0.5

    # 2) evaluate the log-likelihood at the proposed point

    # 3) Metropolis acceptance (uniform prior => the prior term cancels):
    #    accept when  (proposal_ll - current_ll) > log of a uniform(0, 1) draw

    if accept:
        current, current_ll = proposal, proposal_ll
    chain[step] = current

print("acceptance fraction:", np.mean(np.diff(chain) != 0.0))

## Task 2: The same problem with Eryn's `EnsembleSampler`

Now reproduce that result with Eryn's
[`EnsembleSampler`](https://lisa-analysis-tools.github.io/Eryn/user/ensemble.html#eryn.ensemble.EnsembleSampler),
which advances a *population of walkers* with an affine-invariant proposal (no
step-size tuning needed). Build a prior with
[`ProbDistContainer`](https://lisa-analysis-tools.github.io/Eryn/user/prior.html#eryn.prior.ProbDistContainer) of one
[`UniformDistribution`](https://lisa-analysis-tools.github.io/Eryn/user/prior.html) per parameter, seed the sampler
with a [`State`](https://lisa-analysis-tools.github.io/Eryn/user/state.html#eryn.state.State), run it, and read the
samples back with `get_chain()`. Overlay the histogram on the target density.

The `State` holds the full coordinate block of shape
`(ntemps, nwalkers, nleaves, ndim)`; here `ntemps = nleaves = 1`, and
`prior.rvs(size=(1, nwalkers, 1))` appends the `ndim` axis for you. A default
single-branch run stores its chain under the branch name `"model_0"`.

Useful documentation:
* [`EnsembleSampler`](https://lisa-analysis-tools.github.io/Eryn/user/ensemble.html#eryn.ensemble.EnsembleSampler) /
  [`run_mcmc`](https://lisa-analysis-tools.github.io/Eryn/user/ensemble.html#eryn.ensemble.EnsembleSampler.run_mcmc)
* [`State`](https://lisa-analysis-tools.github.io/Eryn/user/state.html#eryn.state.State) /
  [`ProbDistContainer`](https://lisa-analysis-tools.github.io/Eryn/user/prior.html#eryn.prior.ProbDistContainer)
* Informational notebook: see [`06` &sect; A fixed-dimension ensemble sampler](../../06_ErynSmallToLarge.ipynb)

In [ ]:
# imports
from eryn.ensemble import EnsembleSampler
from eryn.state import State
from eryn.priors import ProbDistContainer, UniformDistribution

### Question

How do the number of **walkers** and the length of the **burn-in** change
the posterior you get back?

*Discussion.* More walkers means more independent samples per step and, for the
stretch move, a better-conditioned proposal &mdash; but each walker still needs
to move away from its (possibly poor) starting point before its samples are
representative, which is what burn-in discards. Too short a burn-in leaves the
initial transient in the histogram (biasing it toward the start distribution);
too long wastes samples. Neither walkers nor burn-in change the *target* &mdash;
they change how efficiently and how bias-free you estimate it. The honest amount
of burn-in is set by the chain's autocorrelation time, not guessed (see the
convergence discussion in Task 4).

## Task 3: Parallel tempering across a bimodal target

A single ensemble gets *stuck* when the posterior has well-separated modes:
walkers rarely cross the low-probability valley between peaks, so the mode they
started nearest is over-represented. **Parallel tempering** fixes this by
running copies of the ensemble at decreasing inverse temperatures $\beta$,
sampling the flattened target $\beta\ln\mathcal{L} + \ln p$; hot chains
($\beta\to 0$) roam freely and pass good jumps down to the cold chain
($\beta=1$) you keep.

Run the provided bimodal target **without** tempering (watch it miss a mode),
then **with** tempering (`tempering_kwargs`), and compare histograms. Finally,
estimate the Bayesian **evidence** with thermodynamic integration &mdash; a
tempered run gives it almost for free (freeze the temperature ladder after
burn-in with `stop_adaptation` so the recorded `betas` are constant).

Useful documentation:
* [`EnsembleSampler`](https://lisa-analysis-tools.github.io/Eryn/user/ensemble.html#eryn.ensemble.EnsembleSampler)
  (`tempering_kwargs`) /
  [`TemperatureControl`](https://lisa-analysis-tools.github.io/Eryn/user/temper.html#eryn.moves.tempering.TemperatureControl)
* [`thermodynamic_integration_log_evidence`](https://lisa-analysis-tools.github.io/Eryn/user/utils.html#eryn.utils.utility.thermodynamic_integration_log_evidence)
* Informational notebook: see [`06` &sect; Parallel tempering](../../06_ErynSmallToLarge.ipynb)

In [ ]:
# provided: a two-peaked target (weights 0.8 / 0.2, centres -6 and +6)
from scipy.special import logsumexp

def log_like_bimodal(x):
    return logsumexp([np.log(0.8) - 0.5 * (x[0] + 6.0) ** 2,
                      np.log(0.2) - 0.5 * (x[0] - 6.0) ** 2])

prior1 = ProbDistContainer({0: UniformDistribution(-30.0, 30.0)})
xb = np.linspace(-12.0, 12.0, 400)
target = np.exp([log_like_bimodal(np.array([xi])) for xi in xb])
target /= np.trapezoid(target, xb)
plt.plot(xb, target, "r", lw=1); plt.xlabel("x"); plt.title("bimodal target"); plt.show()

First, a **single-temperature** run. It should over-populate one peak.

Now add **tempering** and rerun. Both modes should appear at the right weights.

Finally, the **evidence** by thermodynamic integration,
$\ln Z = \int_0^1 \langle\ln\mathcal{L}\rangle_\beta\, d\beta$: average the
log-likelihood within each temperature and integrate over the (now frozen)
$\beta$ ladder. We illustrate it on the *unimodal* target from Tasks 1&ndash;2,
where we can **check the result against the exact value** $\ln(1/20)$ for a
unit Gaussian on the prior range $[-10, 10]$ (the same call works on the
bimodal run; a wide, multimodal prior just needs a finer ladder for the same
accuracy).

### Question

What does tempering *buy* you, and what does it cost?

*Discussion.* It buys **mode mixing**: the hot chains flatten the target so
walkers can cross low-probability valleys and carry good jumps down to the cold
chain, so the cold-chain posterior reflects *all* the modes at their correct
relative weights rather than whichever one you started in. As a bonus, the
temperature ladder yields the **evidence** by thermodynamic integration (the
`dlogZ` here is a *discretisation* error from having only a handful of rungs; it
shrinks as you add temperatures, not steps). The cost is compute: you now
advance `ntemps` ensembles instead of one, and only the cold chain is kept as
posterior samples. In the LISA global fit the same tempering runs under the
hood, and model choice usually happens *inside* the sampler via reversible jump
rather than by integrating evidence by hand (informational notebook
[`06` &sect; reversible-jump MCMC](../../06_ErynSmallToLarge.ipynb)).

## Task 4: Recover a GW-flavored signal in noise

Time to sample a *signal*. Inject a monochromatic sinusoid &mdash; a
stand-in for a galactic binary's strain &mdash; into an `FDSignal`, weight it
with a single-channel `LISASens` covariance, and wrap the two in an
`AnalysisContainer` (the object from [`X1`](X1_SensitivitySNR.ipynb)) whose
`eryn_likelihood_function` *is* an Eryn log-likelihood. Recover the three
parameters $(A, f_0, \phi)$ with a **short tempered run**.

The sinusoid generator is provided. Your job: build the injection + container,
then set up priors and run a short tempered sampler over $(A, f_0, \phi)$.
Because the generator regenerates the waveform on every likelihood call, this is
deliberately *small* &mdash; and, as the run is short, **the chain is not
converged** (see the note and Question after it).

Useful documentation:
* [`AnalysisContainer.eryn_likelihood_function`](https://lisa-analysis-tools.github.io/lisa-analysis-tools/user/datacontainer.html#lisatools.analysiscontainer.AnalysisContainer.eryn_likelihood_function)
* [`LISASensSensitivityMatrix`](https://lisa-analysis-tools.github.io/lisa-analysis-tools/user/sensitivity.html#lisatools.sensitivity.LISASensSensitivityMatrix)
* Informational notebook: see [`06` &sect; The lisatools sampler layer](../../06_ErynSmallToLarge.ipynb)
  and [`X1`](X1_SensitivitySNR.ipynb)

In [ ]:
# imports
from lisatools.domains import TDSettings, TDSignal, FDSignal
from lisatools.sensitivity import LISASensSensitivityMatrix
from lisatools.analysiscontainer import AnalysisContainer

In [ ]:
# provided: a sinusoidal strain generator, returned as an FDSignal
dt = 10.0
N = 2 ** 14
td_set = TDSettings(N=N, dt=dt, force_backend="cpu")
t = np.arange(N) * dt

def sinusoid_fd(A, f0, phi):
    """A(sin) monochromatic strain h(t) = A sin(2 pi f0 t + phi), as an FDSignal."""
    h = A * np.sin(2 * np.pi * f0 * t + phi)
    return TDSignal(h[None, :], td_set).fft()

Build the injection at the true parameters, wrap it in a single-channel
`LISASens` `AnalysisContainer`, and check the injected SNR. We set
`likelihood_source_only=True` so the (fixed-PSD) log-likelihood reads out as the
clean source term $-\tfrac12\langle r|r\rangle$.

Now set up 3 priors, a short tempered sampler on `ac.eryn_likelihood_function`, and run.

> **This chain is deliberately short and NOT converged.** With only 100
> recorded steps the marginals below are noisy and depend on the seed; a real
> analysis needs many more steps, a considered burn-in, and a convergence check
> (see the Question). We plot it anyway to see the posterior forming.

### Question

We keep saying this run is *not converged*. What does that mean here, and how
would you check convergence in a real analysis?

*Discussion.* "Converged" means the chain has (a) *forgotten its start* &mdash;
burn-in is past &mdash; and (b) collected enough *effectively independent*
samples that the posterior estimate is stable. A 100-step run fails both: the
walkers are still correlated with their starting points and there are only a
handful of independent draws, so the marginals wobble with the seed. To check it
for real you would: estimate the **autocorrelation time** $\tau$ and require the
run to be many $\tau$ long (and burn several $\tau$); run multiple chains from
different starts and compare them (a **Gelman&ndash;Rubin** $\hat R \to 1$);
watch the running mean/variance and the log-likelihood trace flatten; and, for a
tempered run, confirm the temperature swap-acceptance is healthy. Only then are
the marginals trustworthy. This is the same sampler the global fit runs &mdash;
just far longer, with tuned proposals, and with these diagnostics enforced.

### Where this goes next

You have now taken a likelihood all the way to a posterior: by hand, with an
ensemble, with tempering across modes, and on a real (if tiny) GW signal through
the `AnalysisContainer`. The natural next step is a *variable* number of
sources &mdash; reversible-jump MCMC, where the sampler infers *how many*
signals are present &mdash; which the informational notebook
[`06` &sect; reversible-jump MCMC](../../06_ErynSmallToLarge.ipynb) introduces and
the later `further/` exercises build on. From there, the leap to the full LISA
global fit is the same sampler with specialised proposals &mdash; nothing new
about *how the sampler runs*, only *which moves* it makes.